In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
from pathlib import Path
from tqdm import tqdm

import os
import random
import csv
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import cv2


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = "/content/drive/MyDrive/DLprojekt1/cinic10.zip"
!unzip -q "/content/drive/MyDrive/DLprojekt1/cinic10.zip" -d "/content/"

In [5]:
# global variables and creating folders
DATA_PATH = "/content"
BATCH_SIZE = 128
EPOCHS = 50
NUM_WORKERS = 2

PATIENCE = 10

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

PROJECT_DIR = "/content/drive/MyDrive/DLprojekt1"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
PLOT_DIR = os.path.join(PROJECT_DIR, "plots")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)


## Transforms (single augmentation)

In [6]:
class SobelEdgeBlend:

    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def __call__(self, img):

        img_np = np.array(img)

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        sobel = np.sqrt(sobelx**2 + sobely**2)

        sobel = np.uint8(255 * sobel / np.max(sobel))

        sobel_rgb = np.stack([sobel]*3, axis=-1)

        blended = cv2.addWeighted(img_np, 1-self.alpha, sobel_rgb, self.alpha, 0)

        return Image.fromarray(blended)

def get_transform(augmentation=None):

    transform_list = []

    if augmentation == "crop":
        transform_list.append(transforms.RandomCrop(32,padding=4))

    elif augmentation == "rotation":
        transform_list.append(transforms.RandomRotation(15))

    elif augmentation == "color_jitter":
        transform_list.append(transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1
        ))

    elif augmentation == "blur":
        transform_list.append(transforms.GaussianBlur(3))

    elif augmentation == "sobel":
        transform_list.append(SobelEdgeBlend(alpha=0.5))

    transform_list.extend([

        transforms.ToTensor(),

        transforms.Normalize(
            (0.47889522,0.47227842,0.43047404),
            (0.24205776,0.23828046,0.25874835)
        )

    ])

    return transforms.Compose(transform_list)

def mixup_data(x,y,alpha=0.2):

    if alpha > 0:
        lam = np.random.beta(alpha,alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)

    mixed_x = lam*x + (1-lam)*x[index]

    y_a,y_b = y,y[index]

    return mixed_x,y_a,y_b,lam


## Data loaders

In [7]:
def load_data(augmentation=None, subset_ratio=1.0):

    train_transform = get_transform(augmentation)
    test_transform = get_transform(None)

    train_dataset = datasets.ImageFolder(Path(DATA_PATH)/"train",transform=train_transform)
    val_dataset = datasets.ImageFolder(Path(DATA_PATH)/"valid",transform=test_transform)
    test_dataset = datasets.ImageFolder(Path(DATA_PATH)/"test",transform=test_transform)

    if subset_ratio < 1.0:

        subset_size = int(len(train_dataset)*subset_ratio)
        train_dataset = torch.utils.data.Subset(train_dataset,range(subset_size))

    train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,
                              num_workers=NUM_WORKERS,pin_memory=True)

    val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,
                            num_workers=NUM_WORKERS,pin_memory=True)

    test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,
                             num_workers=NUM_WORKERS,pin_memory=True)

    return train_loader,val_loader,test_loader


## CNN Architecture

In [8]:
class ConvBlock(nn.Module):

    def __init__(self,in_c,out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c,out_c,3,padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self,x):
        return self.block(x)


class CNN(nn.Module):

    def __init__(self,dropout=0.4):
        super().__init__()

        self.block1 = ConvBlock(3,64)
        self.pool1 = nn.MaxPool2d(2)

        self.block2 = ConvBlock(64,128)
        self.pool2 = nn.MaxPool2d(2)

        self.block3 = ConvBlock(128,256)
        self.pool3 = nn.MaxPool2d(2)

        self.block4 = ConvBlock(256,512)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256,10)
        )

    def forward(self,x):

        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.pool3(self.block3(x))
        x = self.block4(x)

        x = self.pool(x)
        x = self.classifier(x)

        return x


## Evaluation

In [9]:

def evaluate(model,loader):

    criterion = nn.CrossEntropyLoss()

    model.eval()

    correct = 0
    total = 0
    loss_total = 0

    with torch.no_grad():

        for images,labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs,labels)

            loss_total += loss.item()

            _,pred = torch.max(outputs,1)

            correct += (pred==labels).sum().item()
            total += labels.size(0)

    acc = correct/total

    return acc,loss_total


## Early Stopping Class

In [10]:
class EarlyStopping:

    def __init__(self,patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def step(self,score):

        if self.best_score is None:
            self.best_score = score

        elif score <= self.best_score:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

        else:

            self.best_score = score
            self.counter = 0

        return self.stop


## Training model

In [11]:
def train_model(model,train_loader,val_loader,optimizer,scheduler,epochs,
                use_mixup=False,name="model"):

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    early_stopping = EarlyStopping(PATIENCE)

    history = {
        "train_loss":[],
        "val_loss":[],
        "val_acc":[]
    }

    best_acc = 0

    model.to(device)

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images,labels in tqdm(train_loader):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            if use_mixup:

                images,y_a,y_b,lam = mixup_data(images,labels)

                outputs = model(images)

                loss = lam*criterion(outputs,y_a) + (1-lam)*criterion(outputs,y_b)

            else:

                outputs = model(images)
                loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()

        train_loss = running_loss / len(train_loader)

        val_acc,val_loss = evaluate(model,val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} TrainLoss:{train_loss:.4f} ValLoss:{val_loss:.4f} ValAcc:{val_acc:.4f}")

        if val_acc > best_acc:

            best_acc = val_acc

            torch.save(
                model.state_dict(),
                os.path.join(CHECKPOINT_DIR,f"best_model_{name}.pt")
            )

        if (epoch+1) % 5 == 0:

            torch.save(
                {
                    "epoch": epoch+1,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "history": history
                },
                os.path.join(CHECKPOINT_DIR,f"checkpoint_{name}_epoch_{epoch+1}.pt")
            )

        if early_stopping.step(val_acc):

            print("Early stopping triggered")
            break

    return history


In [12]:
def save_history_csv(history, experiment_name):

    filename = os.path.join(RESULT_DIR, f"{experiment_name}_history.csv")

    epochs = len(history['train_loss'])

    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss', 'val_acc'])
        for i in range(epochs):
            writer.writerow([
                i+1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['val_acc'][i]
            ])

    print(f"History written in: {filename}")

## Experiment runner

In [13]:
def run_experiment(
    name,
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.4,
    weight_decay=5e-4,
    augmentation=None,
    use_mixup=False
):

    print("Running experiment:", name)

    train_loader,val_loader,test_loader = load_data(augmentation)

    model = CNN(dropout).to(device)


    if optimizer_name == "SGD":

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    elif optimizer_name == "SGD_momentum":

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay
        )

    elif optimizer_name == "Adam":

        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("Unknown optimizer")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    history = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        EPOCHS,
        use_mixup=use_mixup,
        name=name
    )

    test_acc, test_loss = evaluate(model,test_loader)

    print("Test accuracy:", test_acc)

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR,f"{name}.pt")
    )

    save_history_csv(history,name)


    return history,test_acc


## Testing different hiperparameters

#### learning rate:
- LR = 0.1, 0.01, 0.001
- OPTIMIZER = "SGD"
- DROPOUT = 0.3
- WEIGHT_DECAY = 5e-3

In [ ]:
lr_experiments = [

    ("lr_0.1", 0.1, "SGD", 0.3, 5e-3),
    ("lr_0.01", 0.01, "SGD", 0.3, 5e-3),
    ("lr_0.001", 0.001, "SGD", 0.3, 5e-3)

]

lr_results = {}

for name,lr,opt,drop,wd in lr_experiments:

    history,test_acc = run_experiment(

        name=name,
        lr=lr,
        optimizer_name=opt,
        dropout=drop,
        weight_decay=wd,
        augmentation=None,
        use_mixup=False
    )

    lr_results[name] = test_acc

#### optimizer:
- LR = 0.01
- OPTIMIZER = "SGD", "SGD_momentum", "Adam"
- DROPOUT = 0.3
- WEIGHT_DECAY = 5e-3

In [ ]:
optimizer_experiments = [

    ("opt_sgd", 0.01, "SGD", 0.3, 5e-3),
    ("opt_sgd_momentum", 0.01, "SGD_momentum", 0.3, 5e-3),
    ("opt_adam", 0.01, "Adam", 0.3, 5e-3)

]

optimizer_results = {}

for name,lr,opt,drop,wd in optimizer_experiments:

    history,test_acc = run_experiment(

        name=name,
        lr=lr,
        optimizer_name=opt,
        dropout=drop,
        weight_decay=wd,
        augmentation=None,
        use_mixup=False
    )

    optimizer_results[name] = test_acc

#### dropout rate:
- LR = 0.01
- OPTIMIZER = "SGD_momentum"
- DROPOUT = 0.1, 0.3, 0.5
- WEIGHT_DECAY = 5e-3

In [ ]:
dropout_experiments = [

    ("dropout_0.1", 0.01, "SGD_momentum", 0.1, 5e-3),
    ("dropout_0.3", 0.01, "SGD_momentum", 0.3, 5e-3),
    ("dropout_0.5", 0.01, "SGD_momentum", 0.5, 5e-3)

]

dropout_results = {}

for name,lr,opt,drop,wd in dropout_experiments:

    history,test_acc = run_experiment(

        name=name,
        lr=lr,
        optimizer_name=opt,
        dropout=drop,
        weight_decay=wd,
        augmentation=None,
        use_mixup=False
    )

    dropout_results[name] = test_acc

Running experiment: dropout0.1


100%|██████████| 704/704 [00:43<00:00, 16.05it/s]


Epoch 1/20 TrainLoss:1.6547 ValLoss:1036.8801 ValAcc:0.4634


100%|██████████| 704/704 [00:42<00:00, 16.55it/s]


Epoch 2/20 TrainLoss:1.3886 ValLoss:903.6090 ValAcc:0.5470


...

100%|██████████| 704/704 [00:43<00:00, 16.33it/s]


Epoch 19/20 TrainLoss:0.5151 ValLoss:511.7625 ValAcc:0.7836


100%|██████████| 704/704 [00:42<00:00, 16.61it/s]


Epoch 20/20 TrainLoss:0.5142 ValLoss:512.7476 ValAcc:0.7836
Test accuracy: 0.7821444444444444
History zapisane w: /content/drive/MyDrive/DLprojekt1/results/dropout0.1_history.csv


({'train_loss': [1.6547438995066015,
   1.3886286641725085,
   1.2725544580343096,
   1.2011432876302437,
   1.151736785065044,
   1.1106699708510528,
   1.0714839088137855,
   1.0329107339070602,
   0.9911807111718438,
   0.9490391452034767,
   0.9015473172237928,
   0.8463105323978446,
   0.7797028990462422,
   0.7096391704610803,
   0.6288722866130146,
   0.5626660670915787,
   0.5282947346568108,
   0.5183774062686346,
   0.5150755259462378,
   0.5141981455751441],
  'val_loss': [1036.880058735609,
   903.6090496778488,
   770.2156737297773,
   908.7995402961969,
   827.2116258442402,
   744.2995453476906,
   690.4281071275473,
   756.6753181368113,
   627.2109760046005,
   638.8389872312546,
   668.0824946910143,
   602.7578933089972,
   626.6543012261391,
   576.391732826829,
   598.1520651727915,
   546.5624242424965,
   514.8986963331699,
   511.05179357528687,
   511.76250387728214,
   512.7476057708263],
  'val_acc': [0.4634333333333333,
   0.5470222222222222,
   0.6293888888

#### weight decay:
- LR = 0.01
- OPTIMIZER = "SGD_momentum"
- DROPOUT = 0.3
- WEIGHT_DECAY = 1e-2, 1e-3, 1e-4

In [ ]:
weight_decay_experiments = [

    ("wd_1e2", 0.01, "SGD_momentum", 0.3, 1e-2),
    ("wd_1e3", 0.01, "SGD_momentum", 0.3, 1e-3),
    ("wd_1e4", 0.01, "SGD_momentum", 0.3, 1e-4)

]

weight_decay_results = {}

for name,lr,opt,drop,wd in weight_decay_experiments:

    history,test_acc = run_experiment(

        name=name,
        lr=lr,
        optimizer_name=opt,
        dropout=drop,
        weight_decay=wd,
        augmentation=None,
        use_mixup=False
    )

    weight_decay_results[name] = test_acc

Running experiment: weight_decay1e-4


100%|██████████| 704/704 [00:42<00:00, 16.45it/s]


Epoch 1/20 TrainLoss:1.7161 ValLoss:1147.6976 ValAcc:0.4305


100%|██████████| 704/704 [00:41<00:00, 16.88it/s]


Epoch 2/20 TrainLoss:1.4516 ValLoss:903.8869 ValAcc:0.5676


...

100%|██████████| 704/704 [00:41<00:00, 17.09it/s]


Epoch 19/20 TrainLoss:0.5272 ValLoss:559.2072 ValAcc:0.7663


100%|██████████| 704/704 [00:41<00:00, 17.06it/s]


Epoch 20/20 TrainLoss:0.5270 ValLoss:557.0002 ValAcc:0.7669
Test accuracy: 0.7637
History zapisane w: /content/drive/MyDrive/DLprojekt1/results/weight_decay1e-4_history.csv


({'train_loss': [1.7160636674274097,
   1.4515572930262848,
   1.3056121417744593,
   1.208499824628234,
   1.1319224092770706,
   1.0621305926787583,
   1.0009158098731528,
   0.9361882241950794,
   0.87561124173755,
   0.8113097066737034,
   0.7451213830235329,
   0.6798766106367111,
   0.6181363806297834,
   0.5720558063211766,
   0.5457303246313875,
   0.5345253640447151,
   0.5302170679311861,
   0.5283260508863763,
   0.5271537150679664,
   0.5270091389221224],
  'val_loss': [1147.697608526796,
   903.8868607729673,
   745.7153809517622,
   714.2027940303087,
   678.7843055054545,
   613.2480389922857,
   691.4096198827028,
   618.113626241684,
   595.1593076735735,
   580.0866339802742,
   651.3558533340693,
   609.9712839052081,
   605.6273384094238,
   581.2570924088359,
   567.4050686210394,
   556.502577200532,
   557.842998161912,
   557.5315514206886,
   559.2072454690933,
   557.0002279877663],
  'val_acc': [0.4305,
   0.5675777777777777,
   0.6425777777777778,
   0.65355

## Testing different augmentations
#### architecture:
- BEST_LR = 0.01
- BEST_OPTIMIZER = "SGD_momentum"
- BEST_DROPOUT = 0.3
- BEST_WEIGHT_DECAY = 1e-3

In [ ]:
BEST_LR = 0.01
BEST_OPTIMIZER = "SGD_momentum"
BEST_DROPOUT = 0.3
BEST_WEIGHT_DECAY = 1e-3

In [ ]:
run_experiment(

        name="rotation_aug",
        lr=BEST_LR,
        optimizer_name=BEST_OPTIMIZER,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WEIGHT_DECAY,
        augmentation="rotation",
        use_mixup=False
    )

Running experiment: rotation_aug


100%|██████████| 704/704 [00:46<00:00, 15.04it/s]


Epoch 1/40 TrainLoss:1.6387 ValLoss:928.4181 ValAcc:0.5285


100%|██████████| 704/704 [00:50<00:00, 14.05it/s]


Epoch 2/40 TrainLoss:1.3881 ValLoss:815.4847 ValAcc:0.5925


...

100%|██████████| 704/704 [00:46<00:00, 15.30it/s]


Epoch 39/40 TrainLoss:0.5453 ValLoss:489.7684 ValAcc:0.7941


100%|██████████| 704/704 [00:46<00:00, 15.29it/s]


Epoch 40/40 TrainLoss:0.5457 ValLoss:488.3876 ValAcc:0.7942
Test accuracy: 0.7914111111111111
History zapisane w: /content/drive/MyDrive/DLprojekt1/results/rotation_aug_history.csv


({'train_loss': [1.6386780257929454,
   1.3881339339370078,
   1.2877830918878317,
   1.2203982748429885,
   1.17113574992188,
   1.1303521043367006,
   1.0948599574410103,
   1.0671060882847418,
   1.0375856503166936,
   1.0100241761485285,
   0.9866921629926021,
   0.9632513123479757,
   0.9403877538544211,
   0.9191850932653655,
   0.8957693305035884,
   0.874308819797906,
   0.852913820269433,
   0.8319057436998595,
   0.8082042835144834,
   0.7897707448256287,
   0.7670004328882153,
   0.7426158078861508,
   0.715732160278342,
   0.6993279261514544,
   0.6758332777429711,
   0.655393377285112,
   0.6319457427175208,
   0.6176983870735223,
   0.60028256543658,
   0.5874516006389802,
   0.5760385963896458,
   0.5681565950878642,
   0.5621659952131185,
   0.5569619314575737,
   0.5537316653538834,
   0.5504603269086643,
   0.5483413447033275,
   0.546693807552484,
   0.5453016680072654,
   0.5456534422595393],
  'val_loss': [928.4180735051632,
   815.484701693058,
   702.071610778570

In [ ]:
run_experiment(

        name="rotation_aug",
        lr=BEST_LR,
        optimizer_name=BEST_OPTIMIZER,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WEIGHT_DECAY,
        augmentation="rotation",
        use_mixup=False
    )

Running experiment: rotation_aug


100%|██████████| 704/704 [00:46<00:00, 15.04it/s]


Epoch 1/40 TrainLoss:1.6387 ValLoss:928.4181 ValAcc:0.5285


100%|██████████| 704/704 [00:50<00:00, 14.05it/s]


Epoch 2/40 TrainLoss:1.3881 ValLoss:815.4847 ValAcc:0.5925


...

100%|██████████| 704/704 [00:46<00:00, 15.30it/s]


Epoch 39/40 TrainLoss:0.5453 ValLoss:489.7684 ValAcc:0.7941


100%|██████████| 704/704 [00:46<00:00, 15.29it/s]


Epoch 40/40 TrainLoss:0.5457 ValLoss:488.3876 ValAcc:0.7942
Test accuracy: 0.7914111111111111
History zapisane w: /content/drive/MyDrive/DLprojekt1/results/rotation_aug_history.csv


({'train_loss': [1.6386780257929454,
   1.3881339339370078,
   1.2877830918878317,
   1.2203982748429885,
   1.17113574992188,
   1.1303521043367006,
   1.0948599574410103,
   1.0671060882847418,
   1.0375856503166936,
   1.0100241761485285,
   0.9866921629926021,
   0.9632513123479757,
   0.9403877538544211,
   0.9191850932653655,
   0.8957693305035884,
   0.874308819797906,
   0.852913820269433,
   0.8319057436998595,
   0.8082042835144834,
   0.7897707448256287,
   0.7670004328882153,
   0.7426158078861508,
   0.715732160278342,
   0.6993279261514544,
   0.6758332777429711,
   0.655393377285112,
   0.6319457427175208,
   0.6176983870735223,
   0.60028256543658,
   0.5874516006389802,
   0.5760385963896458,
   0.5681565950878642,
   0.5621659952131185,
   0.5569619314575737,
   0.5537316653538834,
   0.5504603269086643,
   0.5483413447033275,
   0.546693807552484,
   0.5453016680072654,
   0.5456534422595393],
  'val_loss': [928.4180735051632,
   815.484701693058,
   702.071610778570

In [ ]:
run_experiment(

        name="blur_aug",
        lr=BEST_LR,
        optimizer_name=BEST_OPTIMIZER,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WEIGHT_DECAY,
        augmentation="blur",
        use_mixup=False
    )

Running experiment: blur_aug


100%|██████████| 704/704 [01:36<00:00,  7.33it/s]


Epoch 1/40 TrainLoss:1.6657 ValLoss:910.5237 ValAcc:0.5402


100%|██████████| 704/704 [01:33<00:00,  7.55it/s]


Epoch 2/40 TrainLoss:1.4131 ValLoss:862.6873 ValAcc:0.5683


...

100%|██████████| 704/704 [01:34<00:00,  7.41it/s]


Epoch 36/40 TrainLoss:0.5156 ValLoss:623.3168 ValAcc:0.7314


100%|██████████| 704/704 [01:33<00:00,  7.53it/s]


Epoch 37/40 TrainLoss:0.5155 ValLoss:623.2797 ValAcc:0.7313
Early stopping triggered
Test accuracy: 0.7283444444444445
History zapisane w: /content/drive/MyDrive/DLprojekt1/results/blur_aug_history.csv


({'train_loss': [1.6656576968221501,
   1.4131448881869966,
   1.3015433825891127,
   1.2249193978919224,
   1.165631740269336,
   1.1103962328792973,
   1.0629633375528185,
   1.0205385325984522,
   0.9759334883737293,
   0.9282415766607631,
   0.880903373895721,
   0.840765886940062,
   0.7975979902866212,
   0.7479977248744532,
   0.7143208956007253,
   0.6755305049432949,
   0.6359566437419165,
   0.6077721048654481,
   0.5766032000326298,
   0.5645359837534752,
   0.5476024431938474,
   0.5355772481892597,
   0.5281165625730698,
   0.5240402106534351,
   0.5222830358384685,
   0.520661037161269,
   0.5195128359747204,
   0.5185028490695086,
   0.5179080257022922,
   0.5174303167414936,
   0.5171986367045478,
   0.5165434990247543,
   0.5171659162945368,
   0.5162567395547574,
   0.515738623301414,
   0.5155924325808883,
   0.515483870594339],
  'val_loss': [910.5236977040768,
   862.6872614175081,
   809.7306871414185,
   753.4834157973528,
   692.9720454961061,
   663.79493060708

In [ ]:
run_experiment(

        name="sobel_aug",
        lr=BEST_LR,
        optimizer_name=BEST_OPTIMIZER,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WEIGHT_DECAY,
        augmentation="sobel",
        use_mixup=False
    )

In [ ]:
run_experiment(

        name="mixup_aug",
        lr=BEST_LR,
        optimizer_name=BEST_OPTIMIZER,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WEIGHT_DECAY,
        augmentation=None,
        use_mixup=True
    )